## Import libraries

In [13]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torchvision
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, dotenv_values 

load_dotenv()

INDEX_DIMENSION=1000
PINECONE_KEY=os.getenv("PINECONE_KEY")
INDEX_NAME=os.getenv("INDEX_NAME")


## EDA (tbd)

Load .csv, .json and images data

In [9]:
styles_metadata = pd.read_csv("../fashion-dataset/styles.csv", delimiter=';')
images_metadata = pd.read_csv("../fashion-dataset/images.csv", delimiter=';')

In [10]:
image_ids = np.sort(styles_metadata.id)
image_address = ["../fashion-dataset/images/"+str(i)+".jpg" for i in image_ids]
image_address_dictionary = {k:v for k,v in zip(image_ids,image_address )}

In [11]:
image_set = {}
missed_items_ids = []
for id in image_address_dictionary:
    address = image_address_dictionary[id]
    try:
        image_set[id] = Image.open(address)
    except:
        missed_items_ids.append(id)

In [12]:
missed_items_ids

[np.int64(12347),
 np.int64(39401),
 np.int64(39403),
 np.int64(39410),
 np.int64(39425)]

# 1. Create Pinecone Index

In [4]:
# initialize Pinecone
pc = Pinecone(api_key=PINECONE_KEY)

In [5]:
# create index
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME, 
        dimension=INDEX_DIMENSION,
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )

index = pc.Index(INDEX_NAME)


In [6]:
index

# 2. Prepare embeddings

In [21]:
# loading pretrained ConvNeXT
model = torchvision.models.convnext_base()

In [22]:
model.classifier

Sequential(
  (0): LayerNorm2d((1024,), eps=1e-06, elementwise_affine=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=1024, out_features=1000, bias=True)
)